# MDG Pipeline — Colab A100

**Role:** Full pipeline — fine-tune all models + run all attack recipes.

**Setup:** Runtime → Change runtime type → A100 GPU + High RAM

⚠️ Colab disconnects after ~12 hours. Re-run cells — the pipeline skips already-done work automatically.

In [ ]:
# ── Cell 1: Clone repo ────────────────────────────────────────────────────────
!git clone https://github.com/YOUR_USERNAME/mdg.git
%cd mdg

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
!pip install -q poetry
!poetry config virtualenvs.create false   # install into Colab's env directly
!poetry install --no-interaction

In [ ]:
# ── Cell 3: Auth tokens ───────────────────────────────────────────────────────
import os
from google.colab import userdata

# Store these in Colab Secrets (key icon on left sidebar)
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# Verify GPU
!nvidia-smi | head -20

In [ ]:
# ── Cell 4: PHASE 1 — Fine-tune all models ────────────────────────────────────
# Skips any model whose checkpoint already exists.
# ~8-12 hours on A100. Re-run safely if disconnected.
!bash mdg/scripts/finetune_all.sh 2>&1 | tee finetune.log

In [ ]:
# ── Cell 5: PHASE 2 — All attack recipes ─────────────────────────────────────
# Skips any model×dataset pair where all_summaries.json already exists.
# ~3-5 days total — run across multiple Colab sessions.
!bash mdg/scripts/attack_all.sh 2>&1 | tee attack.log

In [ ]:
# ── Cell 6: PHASE 3 — Consolidate results ────────────────────────────────────
!python -m mdg.scripts.consolidate_results

In [ ]:
# ── Cell 7: Save results to Google Drive (run after each session) ─────────────
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
SAVE_DIR = '/content/drive/MyDrive/mdg_results'
os.makedirs(SAVE_DIR, exist_ok=True)

# Copy CSVs
for f in ['mdg/finetune/train_results.csv',
          'mdg/adv_attack/attack_results.csv',
          'mdg/results/consolidated.csv']:
    if os.path.exists(f):
        shutil.copy(f, SAVE_DIR)
        print(f'Saved: {f}')

# Copy checkpoints
CKPT_BACKUP = os.path.join(SAVE_DIR, 'checkpoints')
if os.path.exists('mdg/finetune/checkpoints'):
    shutil.copytree('mdg/finetune/checkpoints', CKPT_BACKUP, dirs_exist_ok=True)
    print(f'Checkpoints backed up to {CKPT_BACKUP}')

print('Done. Results safe in Google Drive.')

In [ ]:
# ── Cell 8: Restore from Drive (run at start of NEW session) ─────────────────
# Run this BEFORE Cell 4/5 if you're resuming after a disconnect
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
SAVE_DIR = '/content/drive/MyDrive/mdg_results'

# Restore checkpoints so fine-tuning skips already-done models
CKPT_BACKUP = os.path.join(SAVE_DIR, 'checkpoints')
if os.path.exists(CKPT_BACKUP):
    shutil.copytree(CKPT_BACKUP, 'mdg/finetune/checkpoints', dirs_exist_ok=True)
    print('Checkpoints restored')

# Restore CSVs
for f in ['train_results.csv', 'attack_results.csv', 'consolidated.csv']:
    src = os.path.join(SAVE_DIR, f)
    if os.path.exists(src):
        os.makedirs('mdg/results', exist_ok=True)
        shutil.copy(src, f'mdg/results/{f}')
        print(f'Restored: {f}')

print('Ready to resume.')

In [ ]:
# ── Cell 9: Check progress ────────────────────────────────────────────────────
import os, glob

ckpts = glob.glob('mdg/finetune/checkpoints/*/*/config.json')
print(f'Fine-tuned models : {len(ckpts)} / 33')

summaries = glob.glob('mdg/adv_attack/results/*/*/all_summaries.json')
print(f'Attack runs done  : {len(summaries)} / 33')

for csv_f in ['mdg/finetune/train_results.csv', 'mdg/adv_attack/attack_results.csv']:
    if os.path.exists(csv_f):
        with open(csv_f) as f:
            lines = f.readlines()
        print(f'{csv_f}: {len(lines)-1} rows')